# SolSight — Solder Defect Detection System
### Interactive Demonstration Notebook

This notebook demonstrates the **unsupervised, resolution-agnostic convolutional autoencoder (CAE)** for automatic optical inspection (AOI) of PCBA solder joints using **Structural Dissimilarity (DSSIM)** anomaly heatmaps.

In [ ]:
import sys
from pathlib import Path

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.model.cae import SolderCAE
from src.inference.predict import inspect_patch, create_heatmap_overlay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")

## 1. Load Trained CAE Model & Anomaly Threshold

In [ ]:
checkpoint_path = PROJECT_ROOT / "outputs/checkpoints/best_cae.pt"
config_path = PROJECT_ROOT / "outputs/threshold_config.json"

# Load threshold configuration
with open(config_path, "r") as f:
    threshold_cfg = json.load(f)

global_threshold = threshold_cfg["global_threshold"]
print(f"Global Threshold T: {global_threshold:.4f}")

# Load model
ckpt = torch.load(checkpoint_path, map_location=device)
model = SolderCAE(in_channels=3, base_channels=16).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded SolderCAE from epoch {ckpt['epoch']} (Best Val Loss: {ckpt['best_val_loss']:.4f})")

## 2. Helper Display Function

In [ ]:
def display_inspection(image_path, title="Inspection Result", threshold=global_threshold):
    img = Image.open(image_path).convert("RGB")
    res = inspect_patch(model, img, threshold=threshold, device=device)
    
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
    
    # 1. Original patch
    axes[0].imshow(img)
    axes[0].set_title(f"Input ({img.size[0]}x{img.size[1]}px)", fontweight="bold")
    axes[0].axis("off")
    
    # 2. Reconstruction
    recon_np = res["recon_tensor"][0].permute(1, 2, 0).cpu().numpy()
    axes[1].imshow(np.clip(recon_np, 0.0, 1.0))
    axes[1].set_title("CAE Reconstruction", fontweight="bold")
    axes[1].axis("off")
    
    # 3. DSSIM Heatmap Overlay
    overlay = create_heatmap_overlay(img, res["dssim_map"], alpha=0.55)
    axes[2].imshow(overlay)
    status = "ANOMALY (REJECT)" if res["is_anomaly"] else "NORMAL (PASS)"
    color = "red" if res["is_anomaly"] else "green"
    axes[2].set_title(f"DSSIM Heatmap: {status}\nScore: {res['anomaly_score']:.4f} [T={threshold:.4f}]", color=color, fontweight="bold")
    axes[2].axis("off")
    
    plt.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 3. Exhibit: Golden Reference Normal Solder Joint (128px)

In [ ]:
norm_sample = next((PROJECT_ROOT / "data/synthetic/val_normal/128px").glob("*.png"))
display_inspection(norm_sample, title="Normal Joint Inspection")

## 4. Exhibits: Solder Defects (Voids, Bridging, Cold Joints)

In [ ]:
# Void Defect
void_sample = next((PROJECT_ROOT / "data/synthetic/val_defects/voids/128px").glob("*.png"))
display_inspection(void_sample, title="Defect: Solder Void / Blowhole")

# Bridging Defect
bridge_sample = next((PROJECT_ROOT / "data/synthetic/val_defects/bridging/64px").glob("*.png"))
display_inspection(bridge_sample, title="Defect: Solder Bridging Protrusion")

# Cold Joint Defect
cold_sample = next((PROJECT_ROOT / "data/synthetic/val_defects/cold_joints/64px").glob("*.png"))
display_inspection(cold_sample, title="Defect: Cold / Disturbed Solder Joint")

## 5. Primary Proof Point: Zero-Shot Generalization on Untrained Scale (32px)

The model was trained on native tiers {16, 64, 128} and **never saw a 32px patch during training**.
Here we run inference on unseen 32px patches using the **unmodified model and global threshold `T`**.

In [ ]:
# Untrained 32px Normal
norm_32 = next((PROJECT_ROOT / "data/synthetic/val_normal/32px").glob("*.png"))
display_inspection(norm_32, title="Untrained Scale Generalization: Normal Joint (32px)")

# Untrained 32px Void
void_32 = next((PROJECT_ROOT / "data/synthetic/val_defects/voids/32px").glob("*.png"))
display_inspection(void_32, title="Untrained Scale Generalization: Void Defect (32px)")

## 6. Real-World PCBA Benchmark (Kaggle SolDef_AI)

Zero-shot transfer evaluation on genuine physical PCBA inspection crops from the **SolDef_AI** benchmark dataset.

In [ ]:
# Real Normal Solder Joint
real_norm = next((PROJECT_ROOT / "data/real/normal").glob("*.png"))
display_inspection(real_norm, title="Real PCBA Normal Joint (SolDef_AI Benchmark)", threshold=0.2473)

# Real Defect: Misaligned SMD Chip Joint
real_misaligned = next((PROJECT_ROOT / "data/real/defective/no_good").glob("*.png"))
display_inspection(real_misaligned, title="Real PCBA Defect: Misaligned SMD Component", threshold=0.2473)